# Early Solar System Formation & Planet Migration Analysis

This notebook analyzes the results of our N-body accretion and migration simulations, comparing the **Solar System-like** scenario (low gas density, stable terrestrial growth) with the **Hot Jupiter-like** scenario (high gas density, rapid inward migration).

## Scientific Objectives
1. **Track Accretion History**: Identify the largest planetary embryos and plot their mass growth over time.
2. **Track Orbital Migration**: Plot the distances of these bodies from the central star to demonstrate migration trends.
3. **Compare Disk Clearance**: Analyze the rate of planetesimal clearance (active body count) under different gas disk densities.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100

GM_SUN_KM3_S2 = 1.32712440041279419e11
DAY_IN_S = 86400.0
AU_IN_KM = 149597870.7
GM_CONVERSION = (DAY_IN_S ** 2) / (AU_IN_KM ** 3)
GM_SUN = GM_SUN_KM3_S2 * GM_CONVERSION # ~0.000295912
M_EARTH_SOLAR = 3.003e-6
GM_EARTH = M_EARTH_SOLAR * GM_SUN


In [ ]:
def load_simulation_data(preset):
    # Check local or workspace root path
    path = os.path.join('data', f'simulation_{preset}.npz')
    if not os.path.exists(path):
        path = os.path.join('Projects', 'early-solar-system', 'repo', 'data', f'simulation_{preset}.npz')
    
    if not os.path.exists(path):
        raise FileNotFoundError(f"Cannot find simulation file at: {path}")
        
    data = np.load(path)
    print(f"Loaded preset '{preset}':")
    print(f"  Saved steps: {len(data['t'])}")
    print(f"  Simulation time: {data['t'][-1]/365.25:.1f} years")
    return data


In [ ]:
def analyze_preset(data, preset_name):
    t = data['t'] / 365.25 # Convert to years
    gms = data['gms']
    r = data['r']
    v = data['v']
    num_bodies = gms.shape[1]
    
    # 1. Active body count over time
    active_counts = np.sum(gms > 0, axis=1) - 1 # Exclude the star
    
    # 2. Identify top 4 most massive planets at the end of the simulation
    final_masses_earth = gms[-1, 1:] / GM_EARTH
    top_indices = np.argsort(final_masses_earth)[-4:][::-1] + 1 # +1 to adjust for star offset
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot mass growth of top bodies
    for idx in top_indices:
        mass_history = gms[:, idx] / GM_EARTH
        final_m = final_masses_earth[idx - 1]
        ax1.plot(t, mass_history, label=f'Planet #{idx} (Final: {final_m:.2f} M_earth)', linewidth=2)
        
    ax1.set_title(f'Accretion History: Planet Mass Growth ({preset_name})')
    ax1.set_xlabel('Time (years)')
    ax1.set_ylabel('Mass (Earth Masses)')
    ax1.legend(loc='upper left')
    
    # Plot distance tracking
    star_pos = r[:, 0]
    for idx in top_indices:
        body_pos = r[:, idx]
        dist = np.linalg.norm(body_pos - star_pos, axis=1)
        # Mask out parts where the body is inactive (mass = 0)
        dist_masked = np.where(gms[:, idx] > 0, dist, np.nan)
        ax2.plot(t, dist_masked, label=f'Planet #{idx}', linewidth=2)
        
    ax2.set_title(f'Orbital Distance Tracking ({preset_name})')
    ax2.set_xlabel('Time (years)')
    ax2.set_ylabel('Distance from Star (AU)')
    ax2.set_yscale('log')
    ax2.set_ylim(0.04, 15)
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()
    
    # Print summaries of the top bodies including eccentricity
    print(f"\nSummary for '{preset_name}':")
    for rank, idx in enumerate(top_indices):
        final_m = final_masses_earth[idx - 1]
        final_dist = np.linalg.norm(r[-1, idx] - r[-1, 0])
        
        # Keplerian eccentricity calculation
        pos = r[-1, idx] - r[-1, 0]
        vel_vec = v[-1, idx] - v[-1, 0]
        dist_val = np.linalg.norm(pos)
        v_mag2 = np.dot(vel_vec, vel_vec)
        r_dot_v = np.dot(pos, vel_vec)
        mu = gms[-1, 0]
        
        if mu > 0 and dist_val > 0:
            ecc_vec = ((v_mag2 - mu / dist_val) * pos - r_dot_v * vel_vec) / mu
            final_ecc = np.linalg.norm(ecc_vec)
        else:
            final_ecc = 0.0
            
        print(f"  Rank {rank+1}: Planet #{idx} | Mass: {final_m:.2f} M_earth | Distance: {final_dist:.3f} AU | Eccentricity: {final_ecc:.4f}")


## 2. Solar System-like Simulation Analysis

In [ ]:
data_ss = load_simulation_data('solar_system')
analyze_preset(data_ss, 'Solar System Preset')


## 3. Hot Jupiter-like Simulation Analysis

In [ ]:
data_hj = load_simulation_data('hot_jupiter')
analyze_preset(data_hj, 'Hot Jupiter Preset')


## 4. Planetesimal Disk Clearance Comparison

We compare the clearance rate of the planetesimal disk over time for both simulations. A faster clearance rate indicates more rapid collisions and swallowing of bodies.

In [ ]:
t_ss = data_ss['t'] / 365.25
active_ss = np.sum(data_ss['gms'] > 0, axis=1) - 1

t_hj = data_hj['t'] / 365.25
active_hj = np.sum(data_hj['gms'] > 0, axis=1) - 1

plt.figure(figsize=(10, 5))
plt.plot(t_ss, active_ss, label='Solar System Preset (Low Gas)', color='#4361ee', linewidth=2.5)
plt.plot(t_hj, active_hj, label='Hot Jupiter Preset (Dense Gas)', color='#f85149', linewidth=2.5)
plt.title('Accretion Clearance: Number of Planetesimals Over Time')
plt.xlabel('Time (years)')
plt.ylabel('Active Planetesimals')
plt.legend()
plt.show()


## 5. Radial Mass Distribution Over Time

We analyze how the total mass is distributed radially from the central star at three different epochs: the initial state ($t=0$), the midpoint ($t=400$ years), and the final state ($t=800$ years). This will visually demonstrate whether the solid mass of the system is migrating inward towards the star over time.

In [ ]:
def plot_radial_mass_drift(data, preset_name):
    t = data['t'] / 365.25 # in years
    gms = data['gms']
    r = data['r']
    
    # Select 3 epochs: Start, Middle, End
    steps = [0, len(t) // 2, len(t) - 1]
    colors_epoch = ['#58a6ff', '#bd93f9', '#f85149']
    
    # Radial bins in AU from 0 to 14 AU with 0.5 AU bins
    bin_edges = np.arange(0, 14.5, 0.5)
    
    plt.figure(figsize=(12, 5))
    
    for step, color in zip(steps, colors_epoch):
        epoch_t = t[step]
        epoch_gms = gms[step, 1:] / GM_EARTH # in Earth masses, excluding star
        epoch_pos = r[step, 1:]
        epoch_dist = np.linalg.norm(epoch_pos, axis=1)
        
        # Only include active particles
        active = epoch_gms > 0
        active_dist = epoch_dist[active]
        active_gms = epoch_gms[active]
        
        # Bin masses
        hist, _ = np.histogram(active_dist, bins=bin_edges, weights=active_gms)
        
        # Plot step histogram
        plt.step(bin_edges[:-1], hist, where='post', label=f't = {epoch_t:.1f} yr', color=color, linewidth=2)
        
    plt.title(f'Radial Mass Profile Evolution: {preset_name}')
    plt.xlabel('Distance from Central Star (AU)')
    plt.ylabel('Total Mass in Bin (Earth Masses)')
    plt.xlim(0, 14)
    plt.legend()
    plt.show()

plot_radial_mass_drift(data_ss, 'Solar System Preset (Low Gas)')
plot_radial_mass_drift(data_hj, 'Hot Jupiter Preset (Dense Gas)')


## 6. Mean-Motion Resonance (MMR) Capture and Evolution

Here we identify pairs of growing planets that have entered orbital resonances (where the ratio of their orbital periods is a ratio of small integers like 2:1, 3:2, or 4:3). We then plot the evolution of their period ratios over time to show how they converge and lock into these resonant states during migration.

In [ ]:
def plot_resonance_evolution(data, preset_name):
    t = data['t'] / 365.25 # in years
    gms = data['gms']
    r = data['r']
    
    # Get final masses and identify active bodies
    final_masses = gms[-1]
    active_indices = [i for i in range(1, len(final_masses)) if final_masses[i] > 0]
    active_indices = sorted(active_indices, key=lambda idx: final_masses[idx], reverse=True)[:6]
    
    # Compute distances from star over time
    dists = np.linalg.norm(r - r[:, 0:1], axis=2)
    
    # Sort active indices by final distance
    final_dists = [np.mean(dists[-100:, idx]) for idx in active_indices]
    sorted_pairs = sorted(zip(active_indices, final_dists), key=lambda x: x[1])
    sorted_indices = [x[0] for x in sorted_pairs]
    
    # Find adjacent pairs close to a resonance at the end
    pairs_to_plot = []
    for i in range(len(sorted_indices) - 1):
        idx_inner = sorted_indices[i]
        idx_outer = sorted_indices[i+1]
        
        # Final period ratio
        a_inner = np.mean(dists[-100:, idx_inner])
        a_outer = np.mean(dists[-100:, idx_outer])
        ratio = (a_outer / a_inner) ** 1.5
        
        # Print ratio and check MMR
        for val in [1.25, 1.333, 1.5, 2.0]:
            if abs(ratio - val) < 0.1:
                pairs_to_plot.append((idx_inner, idx_outer))
                break
                
    if not pairs_to_plot:
        print(f"No obvious resonant pairs found for {preset_name} in the top bodies.")
        return
        
    plt.figure(figsize=(12, 5))
    
    for idx_inner, idx_outer in pairs_to_plot:
        period_inner = dists[:, idx_inner] ** 1.5
        period_outer = dists[:, idx_outer] ** 1.5
        ratio = period_outer / period_inner
        
        # Filter steps where both are active
        active_mask = (gms[:, idx_inner] > 0) & (gms[:, idx_outer] > 0)
        
        plt.plot(t[active_mask], ratio[active_mask], 
                 label=f'Planet #{idx_outer} / #{idx_inner} (Final: {ratio[-1]:.3f})', 
                 linewidth=2)
                 
    # Draw horizontal resonance marker lines
    common_res = {2.0: '2:1 MMR', 1.5: '3:2 MMR', 1.333: '4:3 MMR', 1.25: '5:4 MMR'}
    for val, name in common_res.items():
        plt.axhline(y=val, color='gray', linestyle='--', alpha=0.5)
        plt.text(t[0] + 10, val + 0.03, name, color='gray', fontsize=10)
        
    plt.title(f'Orbital Period Ratio Evolution: {preset_name}')
    plt.xlabel('Time (years)')
    plt.ylabel('Period Ratio ($T_{outer} / T_{inner}$)')
    plt.ylim(1.0, 3.0)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.show()

# Plot resonance captures
plot_resonance_evolution(data_ss, 'Solar System Preset (Low Gas)')
plot_resonance_evolution(data_hj, 'Hot Jupiter Preset (Dense Gas)')
